In [62]:
#!pip install selenium
#!pip install dotenv
#!pip install beautifulsoup4
#!pip install pandas

In [63]:
import os #manipulando documentos do diretorio (.env)

from dotenv import load_dotenv #carrega a .env
from selenium import webdriver #driver do seleniu
from selenium.webdriver import ActionChains #Para executar click e escrita como mouse e teclado
from selenium.webdriver.common.by import By #Para identificar os tipos de elementos
from selenium.webdriver.support.ui import WebDriverWait #para criar o tempo de espera para interacoes
from selenium.webdriver.support import expected_conditions as EC #condicao de espera para a interacao

import time #tempo de carregar a pagina para o BS4 ler

from bs4 import BeautifulSoup #para ler o html desejado

import re #para tratar os dados capturados

import pandas as pd #para criacao e analise do dataframe

In [64]:
#Carrega variaveis secretas
load_dotenv()

#Armazena variaveis secretas
usuario = os.getenv("USER_NAME")
password = os.getenv("USER_PASSWORD")
tWait = float(os.getenv("T_WAIT")) #tempo de espera para carregar o .html da pagina, use uma que encaixe com a sua maquina, teste e erro GERALMENTE COM 10 DA CERTO

In [65]:
#Acessa o site RHonline pelo chrome
driver = webdriver.Chrome()
driver.get("https://rhonline.msgas.com.br/#/login")

#CRIANDO ACTION PARA PREENCHER CAMPOS E CLICAR
#site precisa do action para funcionar pois utilizando o driver puro nao foi capaz de escrever
#   estava retornando erro: ElementNotInteractableException
action = ActionChains(driver)

#UTILIZA METODO EXPLICIT WAIT
wait = WebDriverWait(driver, 10)#5 segundos de espera

In [66]:
#--------------------------
#
#   LOGIN NA PAGINA
#
#--------------------------

In [67]:
#identificacao dos campos de interacao
txtUser = wait.until(EC.element_to_be_clickable((By.NAME, "user"))) #campo do usuario
txtPassword = wait.until(EC.element_to_be_clickable((By.NAME, "password"))) #campo de senha
btnEntrar = wait.until(EC.visibility_of_element_located((By.TAG_NAME, "button")))

#Escreve o usuario e senha
action.move_to_element(txtUser).click().send_keys(usuario).perform()
action.move_to_element(txtPassword).click().send_keys(password).perform()

#clica em login
btnEntrar.click()

In [68]:
#--------------------------------------
#
#       Navega ate pagina de pontos
#
#--------------------------------------

In [69]:
#NAVEGAR PARA PAGINA DE PONTO
#   "botao" para clicar-> <span _ngcontent-ng-c3029837786="" aria-label="Acessar menu:  Ponto" class="an an-clock"></span>
ponto = wait.until(EC.element_to_be_clickable((By.XPATH, "//span[contains(@aria-label, 'Ponto')]")))
ponto.click()

#NAVEGA PARA O ESPELHO DE PONTO
#   "botao" para clicar -> <p _ngcontent-ng-c298496229=""> Espelho de ponto </p>
espelho = wait.until(EC.element_to_be_clickable((By.XPATH, "//p[contains(text(), 'Espelho de ponto')]")))
espelho.click()

In [70]:
#----------------------------------------------------------------
#
#   Extrai os dados dos pontos (daquele periodo)
#       (mas para esse teste utilizarei o periodo anterior)
#
#---------------------------------------------------------------

In [71]:
#!!!!!!!!!!aba para ser deletada, apenas para o teste anterior!!!!!!!!!

# opcao para clicar-> <option value="{&quot;initDate&quot;:&quot;2026-04-16T12:00:00Z&quot;,&quot;endDate&quot;:&quot;2026-05-15T12:00:00Z&quot;,&quot;actualPeriod&quot;:true,&quot;id&quot;:&quot;&quot;}"> 16/04/2026 - 15/05/2026 </option>
data_analisada = wait.until(EC.element_to_be_clickable((By.XPATH, "//option[contains(text(), '16/03/2026')]")))
data_analisada.click()

In [72]:
#HORA DO SHOWWW BEATIFULLSOAPPPP
#espera a pagina carregar plenamento para entao extrair o .html
time.sleep(tWait)

page = driver.page_source
soup = BeautifulSoup(page, 'html.parser')
driver.quit() #nao sera mais necessario navegacao

In [73]:
#!!!!!!!!aba [ara ser deletada, apenas para analisar o html mais facilmente
with open('teste.html', 'w', encoding='utf-8') as f:
    f.write(page)

In [74]:
#------------------------------------
#
#   TRATAR OS DADOS DO HTML
#       * coletar horas do ultimo periodo, geralmente esta correto e ja calculado no propio html
#       * coletar dados das horas desse periodo
#
#------------------------------------

In [75]:
#Tratar os dados do html
#Cada linha(tr) possui os dados dos pontos no text, com sujeira junto ;-;
#HORA DA FAXINA! vamos limpar os dados
#   INFORMACOES PARA CRIACAO DO DATAFRAME
#       DIA DO PONTO : ENTRADA 1, SAIDA 1, ENTRADA 2, SAIDA 2
lines = soup.tbody.find_all('tr')
lines[0].get_text(separator=" ") # exemplo de 1 saida, separador para melhorar a visibilidade

'15/04 quarta-feira qua.  07:25   Entrada 1   09:45   Saída 1   13:30   Entrada 2   17:05   Saída 2  Resumo diário Incluir batida Solicitar abono Enviar atestado'

In [76]:
#CRIACAO DAS VARIAVEIS PARA A ANALISE
#Criacao do saldo anterior
Saldo_anterior = soup.h6.text.strip()[1:]

#Criacao do dataframe com o loop
data = []
for line in lines:
    text_line = line.get_text(separator=" ")

    #pega a data com o dia
    match_day = re.search(r"(\d{2}/\d{2})\s*([a-z-áéíóúç.]+)", text_line, re.IGNORECASE)

    if match_day: #se realmente existir dia
        schedule_day = match_day.group(1) #dia do calendario ex: 01/01
        day = match_day.group(2) #dia ex: segunda-feira

        hours = re.findall(r"(\d{2}:\d{2})", text_line)

        item = {"Data": schedule_day, "Dia": day}#linha do dataframe

        #nomeia as batidas de ponto
        for i in range(len(hours)):
            col_name = f"Batida_{i+1}"
            item[col_name] = hours[i]

        data.append(item)

In [77]:
#---------------------------------------------------------
#
#   Analisando os dados para obter
#       * Quantidade de HE realizada
#       * Quantidade de Horas positivas ou negativas do periodo
#       * Quantidade de Horas positivas ou negativas considerando o saldo anterior
#
#-------------------------------------------------------

In [78]:
df = pd.DataFrame(data)
df.head(5)

,Data,Dia,Batida_1,Batida_2,Batida_3,Batida_4
0,15/04,quarta-feira,07:25,09:45,13:30,17:05
1,14/04,terça-feira,07:36,11:31,13:34,18:03
2,13/04,segunda-feira,07:37,11:49,12:56,18:40
3,12/04,domingo,NaN,NaN,NaN,NaN
4,11/04,sábado,NaN,NaN,NaN,NaN


In [79]:
#Tratar os NAN
for col in df.columns[2:]:
    df[col] = df[col].fillna("00:00")
#Converter para timedelta
for col in df.columns[2:]:
    df[col] = pd.to_timedelta(df[col]+":00")

In [80]:
#criando coluna do saldo diario
df["Saldo_diario"] = pd.to_timedelta("00:00:00")
for i in range(1, len(df.columns[2:]), 2):
    col1 = f"Batida_{i}"
    col2 = f"Batida_{i+1}"
    df["Saldo_diario"] += df[col2] - df[col1]

In [81]:
#Aprendendo a criar o array com numpy
#mask = ~df["Dia"].str.lower().isin(["sábado", "domingo"])
 #np.where(mask, pd.to_timedelta("08:00:00"), pd.to_timedelta("00:00:00"))


df["Meta_horas"] = df["Dia"].apply(lambda x: pd.to_timedelta("00:00:00") if x.lower() in ("sábado","domingo") else pd.to_timedelta("08:00:00"))
df["Saldo_Extra"] = df["Saldo_diario"] - df["Meta_horas"]

In [82]:
#formartar essa sujeira do saldo_extra
def formata_timedelta_TO_hh_mm(d):
    segundos = int(d.total_seconds()) #transforma tudo em segundos, aqui ele tira a o -1 day mas -43252352s

    sinal = "-" if segundos < 0 else ""

    #Faz o modulo dos valores
    segundos = abs(segundos)
    #          - (sinal)   08(Horas)  :       00(minutos)
    return f"{sinal}{segundos // 3600}:{(segundos % 3600) // 60}"


In [83]:
df.head(7)

,Data,Dia,Batida_1,Batida_2,Batida_3,Batida_4,Saldo_diario,Meta_horas,Saldo_Extra
0,15/04,quarta-feira,0 days 07:25:00,0 days 09:45:00,0 days 13:30:00,0 days 17:05:00,0 days 05:55:00,0 days 08:00:00,-1 days +21:55:00
1,14/04,terça-feira,0 days 07:36:00,0 days 11:31:00,0 days 13:34:00,0 days 18:03:00,0 days 08:24:00,0 days 08:00:00,0 days 00:24:00
2,13/04,segunda-feira,0 days 07:37:00,0 days 11:49:00,0 days 12:56:00,0 days 18:40:00,0 days 09:56:00,0 days 08:00:00,0 days 01:56:00
3,12/04,domingo,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00
4,11/04,sábado,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00,0 days 00:00:00
5,10/04,sexta-feira,0 days 07:42:00,0 days 11:34:00,0 days 13:21:00,0 days 16:40:00,0 days 07:11:00,0 days 08:00:00,-1 days +23:11:00
6,09/04,quinta-feira,0 days 07:39:00,0 days 11:56:00,0 days 13:33:00,0 days 20:36:00,0 days 11:20:00,0 days 08:00:00,0 days 03:20:00


In [84]:
Saldo_mes = df["Saldo_Extra"].sum()
Saldo_mes = formata_timedelta_TO_hh_mm(Saldo_mes)
Saldo_mes #variavel com o saldo do mês

'-19:17'

In [86]:
Saldo_anterior #variavel com o saldo de horas dos ultimos periodos

'07:51'

In [85]:
Saldo_total = pd.to_timedelta(Saldo_mes + ":00") + pd.to_timedelta(Saldo_anterior + ":00")
Saldo_total = formata_timedelta_TO_hh_mm(Saldo_total)
Saldo_total #variavel com o saldo total

'-11:26'